In [1]:
# Install Unsloth + dependencies (optimized for Qwen2.5)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft bitsandbytes accelerate datasets


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 21.7 MB/s eta 0:00:00


In [2]:
from unsloth import FastLanguageModel
import torch
import gc

# Clean up previous allocations
if 'model' in globals(): del model
if 'tokenizer' in globals(): del tokenizer
gc.collect()
torch.cuda.empty_cache()

model_name = "unsloth/Qwen2.5-7B-Instruct"
max_seq_length = 512  # Further reduced to ensure stability on T4
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceH4/Multilingual-Thinking",
    split="train"
)

dataset = dataset.filter(
    lambda x: x["reasoning_language"] in ["Spanish", "French"]
)

dataset = dataset.remove_columns(
    [c for c in dataset.column_names if c != "messages"]
)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [4]:
print(dataset.column_names)
print(dataset[0])

['messages']
{'messages': [{'content': 'reasoning language: French\n\nYou are an AI chatbot with a lively and energetic personality.', 'role': 'system', 'thinking': None}, {'content': 'Can you show me the latest trends on Twitter right now?', 'role': 'user', 'thinking': None}, {'content': 'Hey there!  While I can\'t check Twitter (X) in real-time or access live data, I can share some tips to help you spot the latest trends:\n\n1. **Open the "Trending" tab** on the Twitter app or website – it updates constantly!  \n2. **Search for hashtags** like #Trending or #Viral to see what’s blowing up.  \n3. **Follow accounts** that curate trends (e.g., @TrendingNow, @ViralThreads).  \n4. **Check regional trends** – they often differ by location!  \n\nRemember, trends are *super fast-moving* and often tied to pop culture, memes, or breaking news. For example, recent trends have included viral challenges (like the "Distracted Boyfriend" meme revival), celebrity drama, or unexpected events (hello, w

In [5]:
def clean_dataset(example):
    convo = example["messages"]

    cleaned = [
        {
            "role": msg.get("role"),
            "content": msg.get("content")
        }
        for msg in convo
        if isinstance(msg, dict)
        and msg.get("role") is not None
        and msg.get("content") is not None
    ]

    return {"messages": cleaned if len(cleaned) > 0 else None}

dataset = dataset.map(clean_dataset)
dataset = dataset.filter(lambda x: x["messages"] is not None)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Filter:   0%|          | 0/400 [00:00<?, ? examples/s]

In [10]:
def formatting_func(example):
    convo = example["messages"]

    if not convo:
        return [""]

    cleaned = []
    for msg in convo:
        if isinstance(msg, dict):
            role = msg.get("role")
            content = msg.get("content")

            if role and content:
                cleaned.append({
                    "role": role,
                    "content": content
                })

    if len(cleaned) == 0:
        return [""]

    # Apply template and ensure truncation to max_seq_length to avoid size mismatch errors
    formatted_text = tokenizer.apply_chat_template(
        cleaned,
        tokenize=False,
        add_generation_prompt=False
    )

    # We return the string; the trainer will handle the tokenization
    # but we can also pre-truncate in the apply_template map step if needed.
    return [formatted_text]

In [7]:
print(formatting_func(dataset[0]))

['<|im_start|>system\nreasoning language: French\n\nYou are an AI chatbot with a lively and energetic personality.<|im_end|>\n<|im_start|>user\nCan you show me the latest trends on Twitter right now?<|im_end|>\n<|im_start|>assistant\nHey there!  While I can\'t check Twitter (X) in real-time or access live data, I can share some tips to help you spot the latest trends:\n\n1. **Open the "Trending" tab** on the Twitter app or website – it updates constantly!  \n2. **Search for hashtags** like #Trending or #Viral to see what’s blowing up.  \n3. **Follow accounts** that curate trends (e.g., @TrendingNow, @ViralThreads).  \n4. **Check regional trends** – they often differ by location!  \n\nRemember, trends are *super fast-moving* and often tied to pop culture, memes, or breaking news. For example, recent trends have included viral challenges (like the "Distracted Boyfriend" meme revival), celebrity drama, or unexpected events (hello, weather disasters!).  \n\nWant me to brainstorm *what* mig

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.4.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
import gc

# Final memory safety check
gc.collect()
torch.cuda.empty_cache()

# Pre-format the dataset with explicit truncation
def apply_template(example):
    # Tokenize and truncate to max_seq_length
    text = formatting_func(example)[0]
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=max_seq_length,
        add_special_tokens=False
    )
    return {"text": tokenizer.decode(tokens["input_ids"])}

dataset = dataset.map(apply_template)

training_args = TrainingArguments(
    output_dir="./qwen2.5-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
    packing=False,
)

# Start training!
trainer.train()

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/400 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)


Step,Training Loss
1,1.914957
2,2.326989
3,1.920166
4,1.679007
5,1.512034
6,1.582395
7,1.232853
8,1.309921
9,1.456855
10,1.073255


TrainOutput(global_step=50, training_loss=1.184041783809662, metrics={'train_runtime': 738.0389, 'train_samples_per_second': 0.542, 'train_steps_per_second': 0.068, 'total_flos': 6375165231891456.0, 'train_loss': 1.184041783809662, 'epoch': 1.0})

### 🧪 Test the Model
Let's test if the model follows the new reasoning patterns in Spanish or French.

In [12]:
from unsloth import FastLanguageModel

# Prepare for inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "¿Cómo puedo mejorar mi productividad personal?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
¿Cómo puedo mejorar mi productividad personal?
assistant
Improving personal productivity involves several strategies that focus on time management, goal setting, and self-care. Here’s a step-by-step guide to help you enhance your efficiency:

---

### **1. Set Clear Goals**
- **SMART Goals**: Make sure your goals are Specific, Measurable, Achievable, Relevant, and Time-bound (e.g., “I will write 500 words of my novel each day for the next month”).
- **Break It Down**: Divide larger tasks into smaller, manageable steps.

---

### **2. Prioritize Tasks**
- **Eisenhower Matrix**: Categorize tasks into four quadr


### 💾 Save and Export Model
We can save the LoRA adapters locally. You can also push them to Hugging Face if you have a token.

In [13]:
# Save the LoRA adapters
model.save_pretrained("qwen2_5_lora_model")
tokenizer.save_pretrained("qwen2_5_lora_model")

# To export to GGUF (for Ollama/Llama.cpp), use:
# model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

print("Model saved to 'qwen2_5_lora_model' folder.")

Model saved to 'qwen2_5_lora_model' folder.


In [14]:
!zip -r model_export.zip qwen2_5_lora_model
print("Model folder zipped as 'model_export.zip'")

  adding: qwen2_5_lora_model/ (stored 0%)
  adding: qwen2_5_lora_model/adapter_config.json (deflated 58%)
  adding: qwen2_5_lora_model/tokenizer_config.json (deflated 43%)
  adding: qwen2_5_lora_model/tokenizer.json (deflated 81%)
  adding: qwen2_5_lora_model/chat_template.jinja (deflated 71%)
  adding: qwen2_5_lora_model/adapter_model.safetensors (deflated 8%)
  adding: qwen2_5_lora_model/README.md (deflated 65%)
Model folder zipped as 'model_export.zip'


### 🔄 How to use the exported model

You currently have **LoRA adapters** (the small files in your zip). To use them in other applications, you typically have two choices:

#### Option A: Merge to 16bit (Standard Transformers)
This combines the base model and your changes into one single folder. This folder can be uploaded to Hugging Face or used in any library that supports Qwen2.5.

In [ ]:
# Merge the LoRA weights into the base model and save as a standalone model
# This requires ~15GB of CPU RAM
model.save_pretrained_merged("qwen2_5_merged_model", tokenizer, save_method = "merged_16bit")
print("Model merged and saved to 'qwen2_5_merged_model'")

### 🚀 Push to Hugging Face Hub
To share your model, you can push it directly to your profile. You will need to get your token from [hf.co/settings/tokens](https://huggingface.co/settings/tokens).

In [16]:
from google.colab import userdata
# Make sure you have 'HF_TOKEN' in your Colab Secrets (the key icon on the left)
try:
    hf_token = userdata.get('HF_TOKEN')

    model.push_to_hub_merged(
        "sarimahsan101/qwen2.5-7b-thinking-esp", # Change to your repo name
        tokenizer,
        save_method = "merged_16bit",
        token = hf_token
    )
    print("Model pushed successfully!")
except Exception as e:
    print(f"Error: {e}. Please ensure you have added your HF_TOKEN to Colab secrets.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...inking-esp/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:  10%|#         |  112MB / 1.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [12:59<00:00, 194.86s/it]


Unsloth: Merge process complete. Saved to `/content/sarimahsan101/qwen2.5-7b-thinking-esp`
Model pushed successfully!


### 📤 Choose your Upload Method

**Option 1: Push only LoRA Adapters (~286MB)**
*Use this if you want a tiny, fast upload and know how to load adapters later.*

In [ ]:
# Push only the small adapter files
try:
    model.push_to_hub(
        "sarimahsan101/qwen2.5-7b-thinking-lora",
        token = userdata.get('HF_TOKEN')
    )
    print("Adapters pushed successfully!")
except Exception as e:
    print(f"Error: {e}")

**Option 2: Push Full Merged Model (~15GB)**
*Use this if you want the model to be 'plug-and-play' for everyone else.*